In [ ]:
!pip install unsloth trl torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 45.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 35.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 64.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 48.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [unsloth]


In [ ]:
import torch, unsloth, transformers, trl, peft
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

print("unsloth", unsloth.__version__)
print("torch", torch.__version__)
print("transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1123 05:20:35.506000 474 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1123 05:20:35.770000 474 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.11.9
torch 2.14.1+cu130
transformers 5.17.0 | trl 1.15.0 | peft 0.21.2
cuda available: True
gpu: Tesla T4


In [ ]:
import unsloth  # Import unsloth first so kernel patches apply cleanly
import re
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

use_bf16 = torch.cuda.is_bf16_supported()
ds = load_dataset("kristiannordby/function_call_sql",split="train")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-1.7B",
    max_seq_length=1024,
    load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
    random_seed=52,
)

def format_row(row):
    msgs = [
        {"role":"system","content":row["system"]},
        {"role":"user","content":row["user"]},
        {"role":"assistant","content":row["sql"]},
    ]
    text =  tokenizer.apply_chat_template(msgs,tokenize=False,enable_thinking=False)
    text = re.sub(r"<think>\s*</think>\s*","",text,flags=re.DOTALL)
    return {"text":text}

ds = ds.map(format_row)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        dataset_num_proc=1,
        max_seq_length=1024,
        packing=False,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=15,
        lr_scheduler_type='cosine',
        fp16=not use_bf16,
        bf16=use_bf16,
        seed=52,
        output_dir="./qwen3-1.7b-sql",
        logging_steps=10,
    )
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

sample_labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
assert len(sample_labels) > 0, "All labels masked! Check response_part delimiter."
print("Unmasked target preview:", tokenizer.decode(sample_labels))

trainer.train()

# save raw
# model.save_pretrained_merged("./Qwen3-1.7B-SQL", tokenizer, save_method="merged_16bit")
# save quantized
model.save_pretrained_gguf("./Qwen3-1.7B-SQL-GGUF", tokenizer, quantization_method="q4_k_m")

README.md:   0%|          | 0.00/339 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 95.2kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1040 [00:00<?, ? examples/s]

==((====))==  Unsloth 2026.11.9: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

... (17 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.10s/it]


Unsloth: Merge process complete. Saved to `/content/Qwen3-1.7B-SQL-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12631-mix-acc0843 (app-b12631-mix-acc0843-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Qwen3-1.7B-SQL-GGUF_gguf/Qwen3-1.7B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed s

{'save_directory': 'Qwen3-1.7B-SQL-GGUF',
 'gguf_directory': 'Qwen3-1.7B-SQL-GGUF_gguf',
 'gguf_files': ['Qwen3-1.7B-SQL-GGUF_gguf/Qwen3-1.7B.Q4_K_M.gguf'],
 'modelfile_location': 'Qwen3-1.7B-SQL-GGUF_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}